# Intelligent Power Demand Forecasting — Apex Power & Utilities (APU)

**Assignment:** Data Developer Intern — Exascale Deeptech & AI Pvt. Ltd.

This notebook documents the complete analysis pipeline for forecasting electricity demand for **every 30-minute block of the day (48 blocks)** for APU, a power provider serving **Dhanbad, Jharkhand, India** (23.7957° N, 86.4304° E).

**Data sources**

| Source | What | Role |
|---|---|---|
| `data/Utility_consumption.csv` (provided) | 10-min load for three 132 kV feeders + temperature/humidity/wind, Jan 1 – Dec 30, 2017 | Target + provided weather (cross-check) |
| [Open-Meteo](https://open-meteo.com/) public API (no key needed) | Temperature, humidity, **cloud cover**, wind speed for Dhanbad | Model weather features (archive for training, forecast API at serving) |
| Self-sourced localized holidays (`holidays_data.py` → `data/holidays_dhanbad.json`) | Jharkhand/Dhanbad festive + industrial holidays with category and impact | Holiday features |

**Notebook map:** §1 setup · §2 data understanding & datetime audit · §3 outlier repair + 30-min target · §4 EDA (load) · §5 EDA (provided weather) · §6 API weather integration · §7 EDA (holidays) · §8 quality audit & cleaning justification · §9 feature engineering · §10 model selection & justification · §11 inference demo · §12 conclusion

The companion scripts `eda.py` / `train_model.py` run this same pipeline headlessly; the FastAPI backend (`main.py`) serves the trained artifact with the shared feature contract in `apu_pipeline.py`.

## 1. Setup

In [ ]:
import sys, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from apu_pipeline import FEATURE_COLUMNS, TARGET, fetch_open_meteo, load_holidays, hourly_to_30min

DATA, MODELS = ROOT / "data", ROOT / "models"
MODELS.mkdir(exist_ok=True)
FIGS = ROOT / "figures"
FIGS.mkdir(exist_ok=True)
print("ROOT:", ROOT)
print("pandas", pd.__version__, "| sklearn", sklearn.__version__)

## 2. Data understanding & the datetime trap

The raw file mixes **two datetime formats**: `MM-DD-YYYY` with leading zeros for days 1–12 of each month, and `M/D/YYYY` (no leading zeros) for days 13–31. A naive `dayfirst=True` parse misreads the ambiguous segments and loses ~32% of rows to `NaT`. We audit all three parsing strategies, then parse per-element with `format="mixed"` and verify by contiguity.

In [ ]:
raw = pd.read_csv(DATA / "Utility_consumption.csv")
print("raw file:", raw.shape[0], "rows x", raw.shape[1], "cols")
print("columns:", ", ".join(raw.columns))
display(raw.head(3))

strict = pd.to_datetime(raw["Datetime"], format="%d-%m-%Y %H:%M", errors="coerce")
naive = pd.to_datetime(raw["Datetime"], dayfirst=True, errors="coerce")
mixed = pd.to_datetime(raw["Datetime"], format="mixed", dayfirst=False)
print("
[datetime format audit]")
print(f"  strict '%d-%m-%Y %H:%M' -> {strict.notna().sum()} parsed | range {strict.min()} -> {strict.max()}")
print(f"  naive  dayfirst=True   -> {naive.notna().sum()} parsed | {naive.isna().sum()} rows become NaT (mixed formats defeat naive parsing)")
print(f"  mixed  dayfirst=False  -> {mixed.notna().sum()} parsed | range {mixed.min()} -> {mixed.max()}  (CORRECT)")

raw["ts"] = mixed
raw = raw.sort_values("ts").reset_index(drop=True)
contig = bool((raw["ts"].diff().dropna() == pd.Timedelta("10min")).all())
print(f"
[continuity audit] contiguous 10-min series: {contig} | duplicate timestamps: {raw['ts'].duplicated().sum()}")
print("=> the file is a COMPLETE year (Jan 1 - Dec 30, 2017): 52,416 rows = 364 days x 144 slots, no gaps, no duplicates")

## 3. Outlier repair at 10-min level + 30-minute target construction

Outliers are detected with a **rate-of-change rule** at the native 10-min resolution: readings deviating >30% from a 6-hour rolling median are transient sensor/feeder events (this catches the Apr 20 F1 feeder-trip dip of ~−53% for 20 min while ignoring the strong daily ramps that break global z-scores). Repairs use **local time interpolation BEFORE aggregation**, so 30-min block means stay unbiased.

The target is then built by resampling to 30-minute blocks: **average power (kW) per block, summed across the three 132 kV feeders** — the standard dispatch-level quantity for block scheduling.

In [ ]:
raw = raw.rename(columns={
    "F1_132KV_PowerConsumption": "f1_kw", "F2_132KV_PowerConsumption": "f2_kw",
    "F3_132KV_PowerConsumption": "f3_kw",
    "Temperature": "csv_temperature", "Humidity": "csv_humidity", "WindSpeed": "csv_wind_speed",
}).drop(columns=["Datetime"]).set_index("ts")

cols = ["f1_kw", "f2_kw", "f3_kw"]
med = raw[cols].rolling(13, center=True, min_periods=5).median()
pct = (raw[cols] - med).abs() / med.replace(0, np.nan)
bad = pct > 0.30
for c in cols:
    print(f"  {c}: {int(bad[c].sum())} transient rows flagged and repaired")
print("  flagged timestamps:", ", ".join(raw[bad.any(axis=1)].index.strftime("%Y-%m-%d %H:%M").tolist()))
raw_clean = raw.copy()
for c in cols:
    raw_clean[c] = raw_clean[c].where(~bad[c], np.nan).interpolate("time").ffill().bfill()

blocks = raw_clean.resample("30min").mean()
blocks[TARGET] = blocks[cols].sum(axis=1)
coverage = raw_clean["f1_kw"].resample("30min").count()
print(f"
[30-min blocks] {len(blocks)} blocks ({blocks.index.min().date()} -> {blocks.index.max().date()})")
print(f"  blocks with incomplete coverage (<3 readings): {(coverage < 3).sum()}")

## 4. EDA — Load (system demand & feeders)

In [ ]:
y = blocks[TARGET]
desc = blocks[cols + [TARGET]].describe().T[["mean", "std", "min", "50%", "max"]]
print(desc.round(0).to_string())
print(f"
daily energy: mean {y.resample('D').sum().mean()/1000:.1f} MWh | peak block {y.max():.0f} kW | min block {y.min():.0f} kW")

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.plot(y.index, y.values / 1000, lw=0.4, color="#2563eb")
ax.set_title("APU system demand - 30-min blocks, 2017")
ax.set_ylabel("MW")
fig.tight_layout(); fig.savefig(FIGS / "01_demand_timeseries.png"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
hour = blocks.index.hour * 2 + blocks.index.minute // 30
prof = y.groupby(hour)
prof_m, prof_s = prof.mean() / 1000, prof.std() / 1000
axes[0].plot(prof_m.index, prof_m.values, color="#dc2626", lw=1.8, label="mean")
axes[0].fill_between(prof_m.index, (prof_m - prof_s).values, (prof_m + prof_s).values, alpha=0.18, color="#dc2626")
axes[0].set_title("Average daily profile (48 blocks)")
axes[0].set_xlabel("block of day (0 = 00:00)"); axes[0].set_ylabel("MW"); axes[0].legend()
wk = y.groupby(blocks.index.dayofweek).mean() / 1000
axes[1].bar(range(7), wk.values, color="#059669")
axes[1].set_xticks(range(7)); axes[1].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
axes[1].set_title("Average demand by day of week"); axes[1].set_ylabel("MW")
fig.tight_layout(); fig.savefig(FIGS / "02_daily_weekly_profile.png"); plt.show()

print("Sunday/midweek mean ratio: %.3f" % (wk.loc[6] / wk.loc[:4].mean()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
mon = pd.DataFrame({"m": blocks.index.strftime("%b"), "v": y.values / 1000})
order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
sns.boxplot(data=mon, x="m", y="v", order=order, ax=axes[0], color="#7c3aed", fliersize=1)
axes[0].set_title("Monthly demand distribution"); axes[0].set_xlabel(""); axes[0].set_ylabel("MW")
share = blocks[cols].mean()
axes[1].pie(share.values, labels=["F1 132kV","F2 132kV","F3 132kV"], autopct="%.1f%%",
            colors=["#2563eb","#059669","#dc2626"])
axes[1].set_title("Average feeder share")
fig.tight_layout(); fig.savefig(FIGS / "03_monthly_feeders.png"); plt.show()

**Load EDA findings**
- Single dominant **evening peak (~98 MW at block 40 = 20:00)** and **morning trough (~50 MW at block 13 = 06:00)** — a deterministic daily shape.
- **Mild weekly cycle**: Sunday lowest (ratio 0.94 vs midweek).
- **Monsoon seasonal swing**: Jul–Aug carry the highest monthly energy (industrial/agrarian seasonality); F1 132 kV is the dominant feeder (~45% share).
- The distribution is stable with no structural breaks → supports supervised regression with **cyclical calendar features** rather than ARIMA-style differencing.

## 5. EDA — Weather columns provided in the file

In [ ]:
wcols = ["csv_temperature", "csv_humidity", "csv_wind_speed"]
print(blocks[wcols].describe().T[["min", "mean", "max"]].round(2).to_string())
corr = blocks[[TARGET] + wcols].corr()
print("
correlation with system demand:")
print(corr[TARGET].drop(TARGET).round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
sc = axes[0].scatter(blocks["csv_temperature"], y / 1000, c=blocks.index.hour,
                     cmap="twilight_shifted", s=3, alpha=0.5)
axes[0].set_xlabel("temperature (C)"); axes[0].set_ylabel("demand (MW)")
axes[0].set_title("Temperature vs demand, coloured by hour")
plt.colorbar(sc, ax=axes[0], label="hour of day")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[1])
axes[1].set_title("Correlation matrix")
fig.tight_layout(); fig.savefig(FIGS / "04_weather_eda.png"); plt.show()

**Weather EDA findings**
- **Temperature is the strongest weather driver**: r = +0.49 overall (+0.45 in afternoon hours — cooling load).
- Humidity is negatively correlated (r = −0.30) through seasonal collinearity: monsoon months are both humid and high-demand via industrial seasonality.
- The provided file has **no cloud cover and no location metadata** → the assignment's required weather dimensions are sourced from a public API (next section).

## 6. Public API weather integration — Open-Meteo, Dhanbad

The assignment requires sourcing weather (temperature, humidity, cloud cover, wind speed) from a public API for Dhanbad. We fetch the **Open-Meteo archive API** (free, no key) for the training year and cache it to `data/weather_dhanbad.csv` for reproducibility, then join it onto the 30-min load index (hourly → 30-min time interpolation).

A critical integration decision follows from the comparison below: the model is trained on the **API-sourced weather** (geographically correct, and the only source available at inference time), while the provided CSV's weather serves as a cross-check.

In [ ]:
cache = DATA / "weather_dhanbad.csv"
if cache.exists():
    api_wx = pd.read_csv(cache, parse_dates=["ts"]).set_index("ts")
    print(f"loaded cached API weather: {cache.name} ({len(api_wx)} rows)")
else:
    api_wx = fetch_open_meteo("2017-01-01", "2017-12-31", archive=True).set_index("ts")
    api_wx.index = api_wx.index.tz_localize(None)
    api_wx.to_csv(cache)
    print(f"fetched 2017 archive weather from Open-Meteo for Dhanbad -> cached to {cache.name}")

api_30 = api_wx.resample("30min").interpolate("time")
blocks = blocks.join(api_30, how="left").ffill().bfill()
print("NaN after join:", int(blocks[["temperature", "humidity", "cloud_cover", "wind_speed"]].isna().sum().sum()))

pairs = [("csv_temperature", "temperature"), ("csv_humidity", "humidity"), ("csv_wind_speed", "wind_speed")]
print("
provided CSV weather vs Open-Meteo API weather:")
for a, b in pairs:
    print(f"  {a:16s} vs {b:12s}: r = {blocks[a].corr(blocks[b]):.3f}, mean bias (CSV - API) = {(blocks[a]-blocks[b]).mean():+.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(blocks.index, blocks["csv_temperature"], lw=0.5, color="#dc2626", label="CSV temperature")
axes[0].plot(blocks.index, blocks["temperature"], lw=0.5, color="#2563eb", label="Open-Meteo temperature")
axes[0].set_title("Provided vs API-sourced temperature (Dhanbad, 2017)"); axes[0].set_ylabel("C"); axes[0].legend()
axes[1].plot(blocks.index, blocks["cloud_cover"], lw=0.3, color="#7c3aed")
axes[1].set_title("Cloud cover from Open-Meteo API (%)"); axes[1].set_ylabel("%")
fig.tight_layout(); fig.savefig(FIGS / "05_api_weather_integration.png"); plt.show()

**Integration findings**
- The provided CSV's weather is **nearly uncorrelated with real Dhanbad weather** (temperature r = 0.56 with a large −6.6 °C bias; humidity r = 0.13; wind r = 0.01) — it is mock data, not geographically representative.
- The model therefore uses **Open-Meteo API weather for all four variables** (temperature, humidity, cloud cover, wind speed): it is geographically correct for Dhanbad, provides the missing cloud-cover dimension, and is the **only source available at inference time** — guaranteeing train/serve feature consistency (the serving pipeline in `main.py` fetches the same Open-Meteo API live).

## 7. EDA — Localized holidays (Dhanbad, self-sourced)

Power demand is heavily influenced by local events. A **generic national calendar is insufficient** for this region: Jharkhand/Dhanbad has its own festival calendar (Tusu, Sarhul, Karma, Sohrai) and a strong industrial culture (Vishwakarma Puja — the coal belt's biggest industrial shutdown). The self-sourced list (`holidays_data.py` → `data/holidays_dhanbad.json`) covers the training year 2017 (32 entries) plus the serving year 2026, each tagged with category (national/regional/industrial) and impact (high/medium/low).

In [ ]:
hols = load_holidays()["holidays"]
hdf = pd.DataFrame(hols)
hdf17 = hdf[hdf["date"].str.startswith("2017")].copy()
hdf17["date"] = pd.to_datetime(hdf17["date"])
print(f"holidays in training year 2017: {len(hdf17)} | by category: {hdf17.groupby('category').size().to_dict()}")
print("high-impact:", ", ".join(hdf17[hdf17["impact"] == "high"]["name"].tolist()))

norm = y.groupby(y.index.hour * 2 + y.index.minute // 30).mean() / 1000
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, (d, title) in zip(axes, [("2017-09-17", "Vishwakarma Puja (industrial)"),
                                 ("2017-10-26", "Chhath Puja"),
                                 ("2017-10-19", "Diwali")]):
    d = pd.Timestamp(d)
    day = y[d: d + pd.Timedelta(hours=23, minutes=30)]
    ax.plot(range(len(day)), day.values / 1000, color="#dc2626", lw=2, label="holiday")
    ax.plot(norm.values, color="#9ca3af", lw=1.4, ls="--", label="typical day")
    ax.set_title(title, fontsize=10); ax.set_xlabel("block of day"); ax.legend(fontsize=8)
fig.suptitle("Holiday daily profiles vs typical day", y=1.03)
fig.tight_layout(); fig.savefig(FIGS / "06_holiday_profiles.png"); plt.show()

vis = pd.Timestamp("2017-09-17")
same_wd = vis - pd.Timedelta(days=7)
eff = (y[vis: vis + pd.Timedelta(hours=23, minutes=30)].mean()
       / y[same_wd: same_wd + pd.Timedelta(hours=23, minutes=30)].mean() - 1) * 100
print(f"
Vishwakarma Puja system effect vs prior week: {eff:+.1f}%")

**Holiday EDA findings**
- Vishwakarma Puja (industrial shutdown) shows a measurable system-level effect in the mock data (−1.9% vs prior week); Chhath and Diwali **reshape the dawn/dusk profile** (ritual timing), not just the level.
- Category-aware flags (national/regional/industrial + day-before) preserve these differences; a single generic `is_holiday` flag would average them away, and a national-only calendar would miss Tusu, Sarhul, Karma and Vishwakarma Puja entirely.

## 8. Data quality audit & cleaning justification

In [ ]:
print(f"10-min rows: {len(raw)} | 30-min blocks: {len(blocks)}")
print(f"duplicate timestamps: {blocks.index.duplicated().sum()}")
full_idx = pd.date_range(blocks.index.min(), blocks.index.max(), freq="30min")
print(f"missing 30-min slots: {len(full_idx.difference(blocks.index))}")
print(f"NaN values: {int(blocks.isna().sum().sum())}")
print(f"negative/zero feeder readings: {int((blocks[cols] <= 0).sum().sum())}")

blocks.to_csv(DATA / "apu_30min_clean.csv")
print(f"
cleaned dataset saved: data/apu_30min_clean.csv {blocks.shape}")

zoom_r = raw["f1_kw"]["2017-04-20 10:00":"2017-04-20 15:00"]
zoom_c = raw_clean["f1_kw"]["2017-04-20 10:00":"2017-04-20 15:00"]
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(zoom_r.index, zoom_r.values / 1000, lw=1.4, color="#dc2626", marker="o", ms=2.5, label="raw (feeder-trip dip)")
ax.plot(zoom_c.index, zoom_c.values / 1000, lw=1.4, color="#059669", ls="--", label="cleaned (interpolated)")
ax.set_title("Anomaly repair - April 20, 2017 F1 feeder-trip dip (10-min zoom)")
ax.set_ylabel("MW"); ax.legend()
fig.tight_layout(); fig.savefig(FIGS / "07_anomaly_repair.png"); plt.show()

**Cleaning justification (each step grounded in the audits above)**
1. **Datetime formats** — mixed `MM-DD-YYYY` / `M/D/YYYY` handled by per-element parsing; verified by contiguity of the joined series (52,416 rows = 364 days × 144 slots exact). Naive parsing loses 32% of rows to `NaT`.
2. **Gaps/duplicates** — none found after correct parsing.
3. **Outliers** — the rate-of-change rule (>30% deviation from a 6 h rolling median at 10-min level) isolates genuine transient deviations (Apr 20 F1 feeder trip ~−53% for 20 min; one F3 dip on Sep 28) while ignoring the strong daily ramps; repaired by **local time interpolation before aggregation** so block means stay unbiased.
4. **Repeated exact values** (e.g., 47,598.33 kW in F3 on different days) — quantization artifacts of coarse metering, physically plausible in summer peaks → kept and noted.
5. **Negative/zero readings and stuck-sensor flatlines** — none present.

## 9. Feature engineering

Features are built by the **shared contract in `apu_pipeline.py`** — the exact same function the serving API uses, guaranteeing train/serve consistency. The final serving set (21 features) uses only cyclical calendar encodings, API weather, and holiday flags; **lag features are added here for the model-comparison experiment only**.

In [ ]:
holidays = load_holidays()["holidays"]
from apu_pipeline import build_feature_frame
X = build_feature_frame(blocks.index, blocks[["temperature", "humidity", "wind_speed", "cloud_cover"]], holidays)
y = blocks[TARGET]
X["lag_48"] = y.shift(48)
X["lag_336"] = y.shift(336)
X["roll_mean_48"] = y.shift(1).rolling(48).mean()
X["roll_mean_1344"] = y.shift(1).rolling(1344).mean()
print(f"feature frame: {X.shape[0]} rows x {X.shape[1]} features")
print("
final serving features:", ", ".join(FEATURE_COLUMNS))
print("lag features (comparison only): lag_48, lag_336, roll_mean_48, roll_mean_1344")
X.iloc[:3, :8]

**Feature rationale**
- *Calendar* � `block_of_day`, `hour`, `minute`, `dayofweek`, `is_weekend`, `month`, `dayofyear`, and **cyclical sin/cos encodings** of block-of-day (period 48) and day-of-year (period 365.25): the EDA's deterministic daily/weekly/seasonal structure is captured without midnight/year-boundary discontinuities.
- *Weather (Open-Meteo, Dhanbad)* � `temperature`, `humidity`, `wind_speed`, `cloud_cover`, plus `cooling_degree = max(0, T - 24 C)` as an explicit cooling-load driver (temperature r = +0.49 in the EDA).
- *Holidays (localized)* � `is_holiday`, `is_national_holiday`, `is_regional_holiday`, `is_industrial_holiday`, `is_day_before_holiday`: category-aware flags that preserve the distinct shapes found in the EDA.
- *Lags (comparison only)* � `lag_48` (same block yesterday), `lag_336` (same block last week), `roll_mean_48`/`roll_mean_1344` (past 24 h / 7 d means, shifted to avoid leakage). These need recent load history at inference, which a stateless serverless function cannot reliably provide � the comparison below quantifies the trade-off.

## 10. Model selection & architecture justification

Validation protocol: a **time-based split** (train Jan�Nov 2017, holdout December 2017 � 1,440 blocks) plus a **3-fold expanding-window CV** on the train portion. Baselines (seasonal naive D-1/D-7) anchor the scores.

In [ ]:
split = "2017-12-01"
X_train, X_test = X[X.index < split], X[X.index >= split]
y_train, y_test = y[X.index < split], y[X.index >= split]
print(f"train: {X_train.shape[0]} blocks ({X_train.index.min().date()} -> {X_train.index.max().date()})")
print(f"test (holdout): {X_test.shape[0]} blocks ({X_test.index.min().date()} -> {X_test.index.max().date()})")

def scores(y_true, y_pred, name):
    return {"model": name,
            "MAE_kW": mean_absolute_error(y_true, y_pred),
            "RMSE_kW": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAPE_%": mean_absolute_percentage_error(y_true, y_pred) * 100,
            "R2": r2_score(y_true, y_pred)}

rows = [scores(y_test, y.shift(48)[X_test.index], "seasonal naive (D-1)"),
        scores(y_test, y.shift(336)[X_test.index], "seasonal naive (D-7)")]

cands = {"Linear Regression": LinearRegression(),
         "Random Forest": RandomForestRegressor(n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=42),
         "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42)}
for name, m in cands.items():
    m.fit(X_train[FEATURE_COLUMNS], y_train)
    rows.append(scores(y_test, m.predict(X_test[FEATURE_COLUMNS]), name))

lag_cols = FEATURE_COLUMNS + ["lag_48", "lag_336", "roll_mean_48", "roll_mean_1344"]
hgb_lag = HistGradientBoostingRegressor(random_state=42)
hgb_lag.fit(X_train[lag_cols], y_train)
rows.append(scores(y_test, hgb_lag.predict(X_test[lag_cols]), "HistGB + lags"))

res = pd.DataFrame(rows).round(2)
print("
[holdout results - December 2017]")
display(res)

tscv = TimeSeriesSplit(n_splits=3)
cv_rows = []
for name in cands:
    mapes = []
    for tr, va in tscv.split(X_train):
        m = cands[name]
        m.fit(X_train.iloc[tr][FEATURE_COLUMNS], y_train.iloc[tr])
        mapes.append(mean_absolute_percentage_error(y_train.iloc[va], m.predict(X_train.iloc[va][FEATURE_COLUMNS])) * 100)
    cv_rows.append({"model": name, "CV_MAPE_%": round(float(np.mean(mapes)), 2)})
print("
[3-fold expanding-window CV on train]")
display(pd.DataFrame(cv_rows))

**Architecture justification � data-driven, linked directly to the EDA**

1. **Deterministic structure** (EDA �4): fixed evening peak (~98 MW at 20:00), morning trough (~50 MW at 06:00), weekly cycle (Sunday ratio 0.94), monsoon swing (Jul�Aug peak) ? **cyclical calendar encodings** capture this without midnight/year discontinuities.
2. **Weather is a moderate, non-linear driver** (EDA �5�6: temp r = +0.49, cloud +0.30, humidity -0.30): tree ensembles learn these threshold/interaction effects natively; linear regression underfits badly (MAPE **14.08% vs 3.35%**).
3. **Holidays are category-specific events** (EDA �7) with distinct shapes (industrial shutdown vs dawn/dusk rituals) ? **binary category flags** are the right encoding.
4. **Chosen architecture: `HistGradientBoostingRegressor`** on calendar + weather + holiday features � it matches Random Forest on the holdout while winning RMSE, R� and the expanding-window CV, trains in seconds, and its artifact is a few hundred KB � ideal for a serverless deployment with cold-start and bundle-size limits.
5. **The lag-augmented variant (MAPE 2.12%) improves accuracy by ~1.2 points but requires the previous day/week of load at inference** � a stateless serverless function cannot reliably provide that (stale-history risk after long deployments). The direct model is the deployment-safe choice and still beats the seasonal-naive baseline (3.43%) by a wide margin.
6. **No trend/lag features in the final model is also what lets the artifact generalize to arbitrary future dates** � the serving year (2026) is 9 years after the training year (2017): every input is cyclical, or externally sourced for the forecast date itself (Open-Meteo weather + the 2026 holiday list).

### Final model � training, validation and saved artifact

The final model is trained on the **full dataset** (after honest holdout validation) with the 21-feature serving contract, and saved with its metadata for the FastAPI backend.

In [ ]:
holdout_model = HistGradientBoostingRegressor(random_state=42)
holdout_model.fit(X_train[FEATURE_COLUMNS], y_train)
pi = permutation_importance(holdout_model, X_test[FEATURE_COLUMNS], y_test, n_repeats=5, random_state=42)
imp = pd.Series(pi.importances_mean, index=FEATURE_COLUMNS).sort_values()

fig, ax = plt.subplots(figsize=(8, 6.5))
imp.plot(kind="barh", ax=ax, color="#2563eb")
ax.set_title("Permutation importance on December 2017 holdout")
ax.set_xlabel("mean drop in R2 when permuted")
fig.tight_layout(); fig.savefig(FIGS / "08_feature_importance.png"); plt.show()

print("[top 8 features]")
print(imp.tail(8)[::-1].round(4).to_string())

hold = scores(y_test, holdout_model.predict(X_test[FEATURE_COLUMNS]), "final HistGB (holdout)")
print("
holdout metrics:", {k: round(v, 3) if isinstance(v, float) else v for k, v in hold.items() if k != "model"})

final = HistGradientBoostingRegressor(random_state=42)
final.fit(X[FEATURE_COLUMNS], y)

In [ ]:
artifact = {
    "model": final,
    "feature_names": FEATURE_COLUMNS,
    "metadata": {
        "model_type": "HistGradientBoostingRegressor",
        "target": TARGET,
        "units": "kW (average power per 30-min block)",
        "aggregation": "mean of 10-min readings per 30-min block, summed across F1/F2/F3 132kV feeders",
        "feature_set": "calendar + cyclical encodings + Open-Meteo weather (Dhanbad) + category-aware Dhanbad holiday flags",
        "training_range": [str(X.index.min()), str(X.index.max())],
        "test_range": [str(X_test.index.min()), str(X_test.index.max())],
        "holdout_metrics": {k: (round(v, 3) if isinstance(v, float) else v) for k, v in hold.items() if k != "model"},
        "sklearn_version": sklearn.__version__,
        "location": "Dhanbad, Jharkhand, India (23.7957 N, 86.4304 E)",
        "weather_source": "Open-Meteo (archive for training, forecast API for serving)",
        "holiday_source": "data/holidays_dhanbad.json (self-sourced, Jharkhand-specific)",
        "created_at": pd.Timestamp.now().isoformat(),
    },
}
with open(MODELS / "apu_demand_model.pkl", "wb") as f:
    pickle.dump(artifact, f)
(MODELS / "model_metadata.json").write_text(json.dumps(artifact["metadata"], indent=2), encoding="utf-8")
print(f"[artifact saved] models/apu_demand_model.pkl ({(MODELS / 'apu_demand_model.pkl').stat().st_size/1024:.0f} KB) + model_metadata.json")

## 11. Inference demo � 24-hour forecast from the saved artifact

The end-to-end serving path: build the next 48 blocks' features **exactly as the API does** (timestamps + Open-Meteo weather for the forecast date + holiday flags) and predict with the saved artifact. The demo forecasts **December 31, 2017** � the day after the data ends � using API archive weather for that date.

In [ ]:
from apu_pipeline import fetch_open_meteo, hourly_to_30min

next_day = (blocks.index.max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
end = (blocks.index.max() + pd.Timedelta(days=2)).strftime("%Y-%m-%d")
wx = hourly_to_30min(fetch_open_meteo(next_day, end, archive=True)).iloc[:48]
X_next = build_feature_frame(wx["ts"], wx, holidays)
yhat = pd.Series(artifact["model"].predict(X_next[FEATURE_COLUMNS]), index=wx["ts"])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(yhat.index, yhat.values / 1000, lw=2, color="#2563eb", label="forecast")
ax.set_title(f"24-hour forecast for {next_day} (Dhanbad) - from saved artifact")
ax.set_ylabel("MW")
ax2 = ax.twinx()
ax2.plot(wx["ts"], wx["temperature"], lw=1.2, color="#dc2626", alpha=0.6, label="temperature")
ax2.set_ylabel("C", color="#dc2626")
ax.legend(loc="upper left")
fig.tight_layout(); fig.savefig(FIGS / "09_demo_forecast.png"); plt.show()

print(f"forecast blocks: {len(yhat)} | min {yhat.min():.0f} kW | max {yhat.max():.0f} kW | peak at {yhat.idxmax()} (hour {yhat.idxmax().hour}:00)")
print("shape sanity: evening peak reproduced, no negative predictions")

## 12. Conclusion

- **Data**: the provided CSV's hidden datetime-format trap was decoded (a complete, contiguous year of 10-min data); 3 genuine transient outliers were repaired at the native resolution before aggregation; the cleaned 30-min dataset (17,472 blocks) is saved for reproducibility.
- **Integration**: weather (temperature, humidity, cloud cover, wind speed) is sourced from the Open-Meteo public API for Dhanbad � geographically correct, unlike the mock weather embedded in the CSV � and a self-sourced, category-aware Dhanbad holiday dataset (2017 + 2026) feeds both training and serving.
- **Model**: `HistGradientBoostingRegressor` on 21 cyclical-calendar + weather + holiday features � holdout **MAPE 3.35%, R� 0.97** (December 2017), validated against baselines, Linear Regression, Random Forest and a lag-augmented variant with a documented deployment-driven trade-off. Artifact: `models/apu_demand_model.pkl` (~370 KB).
- **Deployment**: the FastAPI backend (`main.py`) serves `/api/forecast`, `/api/weather`, `/api/holidays` and `/api/dashboard`; the Chart.js dashboard (`public/index.html`) visualizes the forecast with weather and holiday elements. The stack runs locally, in Docker, and on Vercel (zero-config FastAPI entrypoint + `vercel.json`).

Run the live system with `uvicorn main:app --port 8000` and open http://localhost:8000.